# Lesson 97 · Presenting your work — building it is half the job: the golden structure and the 30-second / 3-minute versions

**Learning goals**
1. Master the "golden structure" for talking about technical work: problem → method → implementation → result → limitations
2. Prepare a 30-second and a 3-minute version of an introduction for each Aurora core module
3. Be able to handle follow-ups like "why not librosa? / why build it from scratch?"
4. Mock interview: record your answer and review it yourself

> Behavioral + technical; golden structure: problem → method → implementation → result → limitations.

← **Previous lesson** · [L96 · Whiteboard practice](L96_whiteboard.ipynb)

> Last lesson was **whiteboard practice**: deriving FFT / CTC / the attention mechanism on the whiteboard, plus a capstone question on the full MFCC pipeline — a rehearsal of the live interview.
> This lesson explores **how to explain clearly what you've built and what you understand**.

## 1. The golden structure for technical interviews

Research-engineer interviews usually include three kinds of question:

| Type | Typical wording | What it assesses |
|---|---|---|
| Concept | "Explain the tradeoff between window size and hop size in an STFT" | Do you really understand it, or did you memorize it? |
| Implementation | "Write an FFT on the whiteboard" / "Write a mel filterbank" | Can you derive and implement it from scratch? |
| Project | "Tell me about your Aurora project" | Can you explain complex work clearly? |

**The golden structure (for project questions)**:
```
① Problem (10 s): what problem I'm solving, and why it's worth solving
② Method (30 s): which technical approach I took, and why I chose it
③ Implementation (60 s): the core code/formula, and the hard parts I ran into
④ Result (20 s): the quantified results (numbers!), and how they compare with a baseline
⑤ Limitations (10 s): where this method falls short, and how I'd improve it next
```
About 2 minutes in total. The interviewer may interrupt at any step with a follow-up — you need to be able to expand any layer.

## 2. The "30-second version" of each Aurora core module

Prepare two versions for every module: a 30-second elevator version + a 3-minute detailed version.
Templates follow; the ✏️ parts are yours to fill in.

In [ ]:
# ✏️ Fill in your 30-second module introductions
# Rules: no more than 3 sentences, including at least 1 number

aurora_30s = {
    "FFT": None,
    # Example:
    # "FFT": (
    #   "I implemented the Cooley-Tukey FFT from scratch, O(N log N) time, "
    #   "with output within 1e-9 of numpy.fft. "
    #   "The core is butterfly divide-and-conquer: N/2 butterfly units per stage, log₂N stages."
    # ),
    "STFT":    None,
    "Mel":     None,
    "MFCC":    None,
    "CTC":     None,
    "Whisper": None,
    "RAG":     None,
}

empty = [k for k, v in aurora_30s.items() if v is None]
print(f'Still to prepare ({len(empty)} modules): {empty}' if empty else 'All 30-second versions ready ✅')

## 3. Common follow-ups and how to approach them

> ⚠️ The numbers in the sample answers below (WER ~4.2%, error 1e-6, about 30× slower, etc.) are **illustrative, not measured in this course**. In an interview, replace them with real numbers you've produced yourself — a number you can't source is more dangerous than no number at all.

**Q: Why not use librosa?**
> It's not that "librosa is bad". Aurora's goal is to prove that I genuinely understand every step —
> whichever line of code the interviewer asks about, I can derive it. An engineer who calls librosa
> and a research engineer who can write a mel filterbank from scratch are showing different levels of understanding.

**Q: How does your implementation compare with what industry uses?**
> The gap is mainly in engineering: my implementation has no CUDA kernels, no streaming,
> and hasn't been load-tested in production. At the algorithm level it's the same —
> my MFCC output matches torchaudio to within 1e-6 under the same parameters.

**Q: If you had to deploy Aurora's ASR to production, how would you do it?**
> First assess the latency requirements: real time (< 500 ms) calls for streaming CTC;
> offline, just use batch Whisper. Then quantize the model (INT8),
> export it with TorchScript or ONNX, and put an audio preprocessing pipeline in front of it.
> Aurora currently covers the feature-extraction layer; the inference layer comes next.

**Q: How do you evaluate your model?**
> For ASR I use WER (word error rate): I fine-tuned Whisper-small
> and measured it on LibriSpeech test-clean, WER ~4.2%. For feature extraction I use cosine similarity
> and downstream classification accuracy as proxy metrics.

## 4. System design: a high-frequency scenario in audio-AI interviews

A common kind of research-engineer interview question: **here's a business scenario — design the complete ML system**.
There's no single right answer; what's being assessed is how complete your thinking is and how aware you are of tradeoffs.

### Classic question: design a real-time meeting transcription system

The interviewer usually expects you to cover these dimensions:

```
① Pin down the requirements
   - Real time or near real time? What's the latency ceiling? (e.g. < 2 s)
   - Single speaker, or multiple speakers to separate (diarization)?
   - Target language? Domain vocabulary (meeting jargon, people's names)?

② Data pipeline
   microphone → 16kHz PCM → pre-emphasis → framing (25ms window / 10ms hop)
   → log-Mel (80 dims) → model input

③ Model choice
   - Real time: Whisper-tiny (39M parameters) + streaming decoding
   - High accuracy: Whisper-large-v3 (1550M), offline batch processing
   - Custom domain: LoRA fine-tuning of Whisper-small

④ Latency breakdown
   audio buffer (500ms) + feature extraction (~5ms) + inference (~100ms)
   → total latency about 600ms, meeting the < 2 s requirement

⑤ Evaluation metrics
   WER (word error rate), RTF (real-time factor < 1.0),
   P95 latency

⑥ Possible improvements
   CTC streaming decoding (no waiting for the end of the sentence), speaker diarization,
   custom-vocabulary hotword injection
```

**Interview tip**: run quickly through every dimension first, then let the interviewer pick one to go deep on.
Don't dive into details straight away — having no overall framework is the most common interview mistake.

## 5. Preparing for coding questions: the hand-written classics of audio AI

Besides whiteboard derivations (L96), interviews will also ask you to write code live in an IDE.
These are the most common kinds of coding question for audio-AI roles:

| Question type | Example | Matching Aurora lesson |
|---|---|---|
| Signal generation | Write `generate_sine(freq, duration, sr)` | L33 |
| Hand-written DFT | The naive O(N²) DFT as a matrix multiplication | L37 |
| Framing | `frame_signal(x, win_len, hop)` | L44 |
| Mel mapping | `hz_to_mel(f)` and `mel_to_hz(m)` | L46 |
| Cosine similarity | `cosine_sim(a, b)` | L80 |
| WER computation | Compute the word error rate with edit distance | L67 |

In [ ]:
# ✏️ Timed live exercise: with no references, write the following two functions in 5 minutes

import numpy as np

def hz_to_mel(f):
    """Convert linear frequency in Hz to the Mel scale."""
    raise NotImplementedError("Implement the HTK mel formula: 2595 * log10(1 + f/700)")

def frame_signal(x, win_len, hop):
    """
    Cut the 1D signal x into frames.
    Returns a 2D array of shape (n_frames, win_len).
    """
    raise NotImplementedError("Implement framing with slicing or np.lib.stride_tricks")

# Automatic check
f_test = np.array([0, 1000, 4000])
mel_ref = 2595 * np.log10(1 + f_test / 700)
assert np.allclose(hz_to_mel(f_test), mel_ref, atol=0.1), "the hz_to_mel formula is wrong"
print("hz_to_mel ✅")

x = np.arange(20, dtype=float)
frames = frame_signal(x, win_len=8, hop=4)
assert frames.shape == (4, 8), f"expected (4,8), got {frames.shape}"
print("frame_signal ✅")

## 6. Behavioral interviews: the STAR structure

Research-engineer roles also ask behavioral questions,
and answering with the STAR structure gives your answers some flesh and blood:

| Letter | Meaning | What to say |
|---|---|---|
| **S**ituation | Background | Which project, at what point in time |
| **T**ask | Task | What your specific responsibility was |
| **A**ction | Action | The concrete steps you took (say "I", not "we") |
| **R**esult | Result | The quantified result, or what you learned |

**Common behavioral questions in an Aurora setting:**

- "Tell me about a technical difficulty you ran into and how you solved it"
  → you could use a numerical-stability issue in your FFT implementation, or the boundary conditions of the CTC forward algorithm
- "Tell me about an experiment that didn't meet expectations"
  → you could use a Whisper fine-tune where WER didn't improve, and your analysis of why
- "How do you ensure code quality?"
  → every Aurora function has assert tests pinning it to a numpy reference implementation

## 7. At the end of the interview: the questions you should ask

When the interviewer says "Do you have any questions for us?" — that's an opportunity, not a formality.
A good question shows the interviewer that you're really thinking hard about the job itself.

**Examples of good questions:**

- "Which part of your team's current tech stack would you most like to see improve significantly over the next year?"
- "Since you joined this team, what challenge has gone most beyond what you expected?"
- "What are the most important deliverables in the first 3 months for a new hire?"
- "Roughly how does the team split its time between research and engineering?"

**Not-so-good questions:**
- "What's the salary?" (save it for the HR stage)
- "How much vacation do you get?" (wrong focus)
- "No, I don't have any questions" (a missed opportunity)

## Exercise: mock interview

Record 2 minutes on your phone answering the question below, then review yourself against the scoring criteria.

**The question**: Tell me about your Aurora project — in particular, how you implemented the FFT,
and why you built it from scratch instead of calling numpy.fft.

In [ ]:
# Self-review sheet (fill it in after recording)
mock_interview = {
    "duration_seconds":     None,   # how many seconds did you talk?
    "mentioned_numbers":    None,   # did you mention any numbers/results? True/False
    "explained_why_scratch": None,  # did you explain clearly why you built it from scratch? True/False
    "covered_limitation":   None,   # did you mention the limitations? True/False
    "filler_words_count":   None,   # count how many times "um / like / you know" came up
    "next_improvement":     None,   # what will you improve next time?
}

score = sum(1 for k in ["mentioned_numbers","explained_why_scratch","covered_limitation"]
            if mock_interview.get(k) is True)
if all(v is not None for v in mock_interview.values()):
    print(f'Key-item score: {score}/3')
    if score == 3:
        print('✅ All the core elements are there — keep polishing your fluency')
    else:
        print('⚠️  Some elements are missing — reread the golden structure and record again')
else:
    print('Do the recording and self-review first, then run this cell')

## 8. The 3-minute version: the FFT as an example

The 30-second version lets people know what you did; the 3-minute version makes them believe you really understand it.
Taking the FFT as the example, here's the structure of a 3-minute version:

**(0:00-0:20) Background**
"The FFT is the core of Aurora's whole audio-processing chain — STFT, Mel and MFCC all depend on it.
I chose to implement it from scratch rather than call numpy.fft so that I could derive every step on a whiteboard."

**(0:20-1:20) Method**
"Cooley-Tukey divide and conquer: split an N-point DFT into two N/2-point DFTs, odd and even, and recurse until N=1.
Each stage has N/2 butterfly units, and each unit does one complex multiplication (the twiddle factor) plus two complex additions.
There are log₂N stages, so the total complexity is O(N log N)."

**(1:20-1:50) Implementation and verification**
"The core is the `butterfly(E, O, k, N)` function: it takes two N/2-point complex DFTs
and outputs `E + W·O` and `E - W·O`, where W = e^{-2πik/N}.
I verify it with `np.allclose(my_fft(x), np.fft.fft(x), atol=1e-9)` — the error is within 1e-9."

**(1:50-2:10) Limitations**
"My implementation is pure Python with no SIMD/CUDA optimization;
at N=4096 it's about 30× slower than numpy.fft (an illustrative number, not measured in this course — go by your own benchmark). In industry you'd use FFTW or cuFFT."

**(2:10-3:00) Taking follow-ups**
Leave time for people to ask "so why not just use numpy.fft?" —
you already planted the answer in your first sentence, so now you just expand on it.

## 9. Extra preparation for remote interviews

Most first-round interviews now happen online, so on top of the technical questions there's your setup to think about.

### Technical setup

- **Camera**: look into the lens, not at the screen — practice the habit beforehand
- **Microphone**: use a headset mic to avoid echo; check the sound quality before the interview with `audacity` or your system recorder
- **Network**: wired beats wireless; run a speed test beforehand and make sure upload is > 5 Mbps
- **Screen sharing**: turn off notifications (macOS: Do Not Disturb), and open the files/terminal you plan to show in advance

### Coding environment

Have a clean IDE window ready before the interview:
- Bump up the font size (the interviewer is watching your screen; the text needs to be clear)
- Turn off Copilot and other AI completion (the interviewer wants to see your own thinking)
- Have a terminal ready that can run Python straight away

### Handling a glitch during the interview

If the connection stutters or you can't hear clearly, just say "Sorry, the connection's a bit choppy — could you repeat that?"
That's far better than pretending you understood and then answering the wrong question.

## 10. ✏️ Prepare 3 complete STAR stories

Write them out before the interview — not to memorize them, but to **make sure each story's skeleton is complete**.
Once they're written down, you'll find that many stories you thought "I can tell this clearly" are actually missing a quantified result.

In [ ]:
# ✏️ Write out 3 STAR story frameworks (any length)
star_stories = {
    "Story 1: a technical difficulty": {
        "situation":  None,   # which project, at what point in time
        "task":       None,   # what your responsibility was
        "action":     None,   # the concrete steps you took ("I", not "we")
        "result":     None,   # the quantified result, or what you learned
    },
    "Story 2: a failure / change of course": {
        "situation":  None,
        "task":       None,
        "action":     None,
        "result":     None,
    },
    "Story 3: improving something on your own initiative": {
        "situation":  None,
        "task":       None,
        "action":     None,
        "result":     None,
    },
}

for story_name, parts in star_stories.items():
    filled = sum(1 for v in parts.values() if v is not None)
    complete = "✅" if filled == 4 else f"⬜ {filled}/4"
    print(f"{complete}  {story_name}")
    if filled < 4:
        missing = [k for k, v in parts.items() if v is None]
        print(f"       missing: {missing}")

## 11. What different employers emphasize

The shape of a research-engineer interview varies with the type of employer:

| Employer type | Typical rounds | Emphasis |
|---|---|---|
| Big-tech speech teams (Google, Meta) | 5-6 rounds: algorithms + system design + behavioral + paper discussion | Theoretical depth + engineering at scale |
| Audio-AI startups | 2-4 rounds: take-home + technical + culture fit | Can you get up to speed and ship quickly? |
| University labs | 1-2 rounds: paper discussion + a chat about research directions | Research thinking + publication potential |
| Consultancies / service firms | 1-2 rounds: questions built on real project scenarios | Engineering ability to solve real problems |

**The interview types Aurora maps onto most directly**: audio-AI startups (a take-home where you show Aurora's code)
and big-tech technical rounds (deriving the FFT/MFCC on a whiteboard).
A big-tech system-design round might ask you to "design the complete backend for a voice assistant" — answer it with the pipeline framework from L92.

## Wrap-up

Explaining clearly what you understand is a skill in its own right; an interview is just one place it gets tested — working with colleagues, writing documentation, even explaining things to your future self all call on it too.
If you can explain each module clearly in 2 minutes, that shows you truly understand it, rather than having only memorized it.

**Next lesson L98**: the course retrospective — what you did in 6 months, what's still missing, and where Aurora v2 goes from here.

---

→ **Next lesson** · [L98 · Retrospective](L98_retrospective.ipynb)

> Next lesson is the **retrospective**: a look back at six months of milestones, a summary of what worked, and putting numbers on your learning curve.